# Suppressed-Carrier AM: Modulate, Demodulate & Recover a Message

*Python/Colab conversion of the MATLAB script `AM.m`*
(from **Software Receiver Design**, Johnson, Sethares & Klein).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oalnaseri/CommSystems_Course/blob/main/AM.ipynb)

> **How to run:** Click the **Open in Colab** badge above, then run each cell top-to-bottom with `Shift + Enter`, or use **Runtime -> Run all**.

**The idea:** a message is multiplied by a 1000 Hz carrier (**modulation**), sent, then multiplied by the same carrier again (**demodulation**), and finally cleaned up with a **low-pass filter** to recover the original message. This is the complete transmit → receive chain.

No installation is needed — `numpy`, `scipy`, `matplotlib` and `plotly` are pre-installed in Colab.

## 1. Setup

Import numerical, signal-processing, and plotting libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import remez, lfilter   # remez = MATLAB's firpm; lfilter = MATLAB's filter

# Interactive plotting (pre-installed in Colab). Install locally if missing.
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plotly'])
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

%matplotlib inline
plt.rcParams['figure.figsize'] = (9, 4)

## 2. Helper functions for plotting (with optional zoom)

To plot **time-domain** waveforms, a `plot_signal` helper is defined , plus an interactive Plotly version so you can **zoom the plot themselves**.

* `plot_signal(t, x, title)` → full time view
* `plot_signal(t, x, title, tlim=(0, 0.1))` → zoom the time axis
* `plot_signal_interactive(...)` → drag-to-zoom / pan / hover

In [ ]:
def plot_signal(t, x, title='', ylabel='amplitude', tlim=None, ylim=None):
    """Static time-domain plot (mirrors MATLAB plot + axis + title)."""
    fig, ax = plt.subplots()
    ax.plot(t, x)
    ax.set_xlabel('time (seconds)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True)
    if tlim is not None:
        ax.set_xlim(tlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    fig.tight_layout(); plt.show()


def plot_signal_interactive(t, x, title='', tlim=None):
    """Interactive time-domain plot: drag-to-zoom, pan, hover."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t, y=x, mode='lines', name='signal'))
    fig.update_xaxes(title_text='time (seconds)')
    fig.update_yaxes(title_text='amplitude')
    if tlim is not None:
        fig.update_xaxes(range=list(tlim))
    fig.update_layout(height=380, title_text=title, hovermode='x unified')
    fig.show()

A `plotspec` pair is also added (waveform + magnitude spectrum) so you can *see* the frequency shifting that AM performs — handy for understanding **why** the low-pass filter recovers the message.

In [ ]:
def plotspec(x, Ts, flim=None, title=''):
    """Waveform + magnitude spectrum (Python port of textbook plotspec.m)."""
    x = np.asarray(x); N = len(x)
    t = Ts * np.arange(1, N + 1)
    ssf = np.arange(-N/2, N/2) / (Ts * N)
    fxs = np.fft.fftshift(np.fft.fft(x))
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6))
    ax1.plot(t, x); ax1.set_xlabel('seconds'); ax1.set_ylabel('amplitude')
    ax1.set_title(title or 'Waveform'); ax1.grid(True)
    ax2.plot(ssf, np.abs(fxs)); ax2.set_xlabel('frequency (Hz)')
    ax2.set_ylabel('magnitude'); ax2.grid(True)
    if flim is not None:
        lo, hi = (-flim, flim) if np.isscalar(flim) else (flim[0], flim[1])
        ax2.set_xlim(lo, hi)
    ax2.set_title('Magnitude spectrum')
    fig.tight_layout(); plt.show()


def plotspec_interactive(x, Ts, flim=None, title=''):
    """Interactive waveform + magnitude spectrum (Plotly)."""
    x = np.asarray(x); N = len(x)
    t = Ts * np.arange(1, N + 1)
    ssf = np.arange(-N/2, N/2) / (Ts * N)
    fxs = np.fft.fftshift(np.fft.fft(x))
    fig = make_subplots(rows=2, cols=1,
                        subplot_titles=('Waveform (time domain)', 'Magnitude spectrum'))
    fig.add_trace(go.Scatter(x=t, y=x, mode='lines'), row=1, col=1)
    fig.add_trace(go.Scatter(x=ssf, y=np.abs(fxs), mode='lines'), row=2, col=1)
    fig.update_xaxes(title_text='seconds', row=1, col=1)
    fig.update_yaxes(title_text='amplitude', row=1, col=1)
    fig.update_xaxes(title_text='frequency (Hz)', row=2, col=1)
    fig.update_yaxes(title_text='magnitude', row=2, col=1)
    if flim is not None:
        lo, hi = (-flim, flim) if np.isscalar(flim) else (flim[0], flim[1])
        fig.update_xaxes(range=[lo, hi], row=2, col=1)
    fig.update_layout(height=650, showlegend=False, title_text=title, hovermode='x unified')
    fig.show()

## 3. Build the AM system

**Signals:**
* **`w`** — the message: a slow rising ramp *plus* a 20 Hz cosine.
* **`c`** — the 1000 Hz carrier.
* **`v = c · w`** — the transmitted (modulated) signal.
* **`x = v · c2`** — the demodulated signal (multiply by the carrier again).
* **`m`** — the recovered message after low-pass filtering.

**Note on `firpm`:** MATLAB's optimal (Parks–McClellan) FIR designer is `scipy.signal.remez`. MATLAB normalises band edges to the Nyquist frequency (=1), so we call `remez(..., fs=2)` to make SciPy's Nyquist equal 1 as well — giving an identical filter.

In [ ]:
time = 0.3          # length of time (seconds)
Ts = 1/10000        # sampling interval  ->  10 kHz sample rate
t = np.arange(Ts, time + Ts, Ts)     # time vector: Ts:Ts:time
lent = len(t)

fm = 20             # message tone frequency (Hz)
fc = 1000           # carrier frequency (Hz)
c = np.cos(2*np.pi*fc*t)                       # carrier

# create a message = rising ramp + 20 Hz cosine
w = 5/lent * np.arange(1, lent + 1) + np.cos(2*np.pi*fm*t)

v = c * w                                      # modulate with carrier

c2 = np.cos(2*np.pi*fc*t)                       # carrier copy for demod
x = v * c2                                      # demodulate received signal

# --- Low-pass filter design (MATLAB firpm -> scipy remez) ---
fbe = [0, 0.1, 0.2, 1]        # band edges (normalised to Nyquist = 1)
damps = [1, 0]                # desired gain per band (pass, stop)
fl = 100                      # filter order
b = remez(fl + 1, fbe, damps, fs=2)   # fs=2 -> Nyquist=1, matching MATLAB

m = 2 * lfilter(b, 1, x)               # LPF the demodulated signal

print(f'sample rate = {1/Ts:.0f} Hz, N = {lent} samples, filter taps = {len(b)}')

## 4. The four-panel figure  *(exact MATLAB reproduction)*

This recreates the original figure: message **(a)**, after modulation **(b)**, demodulated **(c)**, and the recovered message **(d)** — all zoomed to the first 0.1 s with the same axis limits as `AM.m`.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 10))

axes[0].plot(t, w);  axes[0].axis([0, 0.1, -1, 3])
axes[0].set_ylabel('amplitude'); axes[0].set_title('(a) message signal')

axes[1].plot(t, v);  axes[1].axis([0, 0.1, -2.5, 2.5])
axes[1].set_ylabel('amplitude'); axes[1].set_title('(b) message after modulation')

axes[2].plot(t, x);  axes[2].axis([0, 0.1, -1, 3])
axes[2].set_ylabel('amplitude'); axes[2].set_title('(c) demodulated signal')

axes[3].plot(t, m);  axes[3].axis([0, 0.1, -1, 3])
axes[3].set_ylabel('amplitude')
axes[3].set_title('(d) recovered message is a LPF applied to (c)')

for ax in axes:
    ax.set_xlabel('time (seconds)'); ax.grid(True)
fig.tight_layout(); plt.show()

## 5. (a) Message signal `w`

A slow rising ramp with a 20 Hz cosine riding on top — this is what we want to transmit and recover.

In [ ]:
plot_signal(t, w, title='(a) message signal w', tlim=(0, 0.1), ylim=(-1, 3))

**Interactive** — drag to zoom, hover to read values:

In [ ]:
plot_signal_interactive(t, w, title='(a) message signal w', tlim=(0, 0.1))

## 6. (b) After modulation `v = c · w`

Multiplying by the 1000 Hz carrier fills the waveform with fast oscillations whose *envelope* follows the message. In the spectrum, the message energy has moved up near **±1000 Hz**.

In [ ]:
plot_signal(t, v, title='(b) message after modulation', tlim=(0, 0.1), ylim=(-2.5, 2.5))

**Spectrum** (full, then zoomed to ±1500 Hz) shows the shift to the carrier:

In [ ]:
plotspec(v, Ts, flim=1500, title='(b) modulated signal v')

**Interactive** time view and spectrum:

In [ ]:
plot_signal_interactive(t, v, title='(b) modulated signal v', tlim=(0, 0.1))

In [ ]:
plotspec_interactive(v, Ts, flim=1500, title='(b) modulated signal v')

## 7. (c) Demodulated signal `x = v · c2`

Multiplying by the carrier a second time shifts the spectrum back down: part of the energy returns to **baseband** (near 0 Hz — the message we want) and part lands near **±2000 Hz** (unwanted, to be filtered out).

In [ ]:
plot_signal(t, x, title='(c) demodulated signal', tlim=(0, 0.1), ylim=(-1, 3))

**Spectrum** — note the wanted low-frequency content *and* the high-frequency image near 2000 Hz:

In [ ]:
plotspec(x, Ts, flim=2500, title='(c) demodulated signal x')

**Interactive**:

In [ ]:
plot_signal_interactive(t, x, title='(c) demodulated signal x', tlim=(0, 0.1))

In [ ]:
plotspec_interactive(x, Ts, flim=2500, title='(c) demodulated signal x')

## 8. (d) Recovered message `m` (low-pass filtered)

The low-pass filter keeps only the baseband part of (c), removing the 2000 Hz image. The factor of **2** compensates for the amplitude lost in demodulation. The result **`m`** closely matches the original message **`w`** — the AM system works! (The brief distortion at the very start is the filter's start-up transient.)

In [ ]:
plot_signal(t, m, title='(d) recovered message m', tlim=(0, 0.1), ylim=(-1, 3))

**Interactive**:

In [ ]:
plot_signal_interactive(t, m, title='(d) recovered message m', tlim=(0, 0.1))

## 9. Did it work? Overlay original vs. recovered

Plotting `w` and `m` together shows how faithfully the message was recovered.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t, w, label='original message w', linewidth=1.5)
ax.plot(t, m, label='recovered message m', linewidth=1.2, alpha=0.8)
ax.set_xlim(0, 0.1); ax.set_ylim(-1, 3)
ax.set_xlabel('time (seconds)'); ax.set_ylabel('amplitude')
ax.set_title('Original vs. recovered message'); ax.legend(); ax.grid(True)
fig.tight_layout(); plt.show()

## 10. Notes & tips

- **The big picture:** modulation (× carrier) moves the message up to the carrier frequency; demodulation (× carrier again) brings it back to baseband but leaves a high-frequency image; the low-pass filter removes that image, recovering the message. The `2*` restores its amplitude.
- **`firpm` → `remez`:** use `scipy.signal.remez(numtaps, bands, desired, fs=2)`. `numtaps = fl + 1`, and `fs=2` makes SciPy's Nyquist match MATLAB's normalised edges.
- **`filter` → `lfilter`:** MATLAB `filter(b,1,x)` is `scipy.signal.lfilter(b, 1, x)`.
- **Static zoom:** pass `tlim=(0, 0.1)` to `plot_signal`, or `flim=` to `plotspec`.
- **Interactive zoom:** use the `_interactive` functions, drag a box to zoom, double-click to reset.
- **Try it yourself:** change `fm`, `fc`, or the filter band edges `fbe` and re-run to see how recovery quality changes.